# Join Track Vectors to Show Vectors and Estimate Show Track List

## Load useful libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

In [ ]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import FloatType

## User settings

In [ ]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 300
spark_memory = '70G'
percentile_cutoff = 0.8
approx_quantile_precision = 0.05
irq_min = 45.
irq_max = 60. * 11.

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

## Initialize Spark session

In [ ]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

## Load show and track library data

In [ ]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show = spark.read.parquet(path_show_output)

In [ ]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library = spark.read.parquet(path_library_output)

## Join the show and track library dataframes

In [ ]:
sdf_cross_joined = (
    sdf_show
    .crossJoin(sdf_library)
    .orderBy('time_step', 'id')
)

In [ ]:
sdf_cross_joined.show(3)

## Define a function for computing cosine similarity

In [ ]:
@F.udf(returnType=FloatType())
def compute_cosine_similarity(vector1, vector2):
    cosine_dist = cosine(np.array(vector1), np.array(vector2))
    similarity_score = 1 - cosine_dist
    return float(similarity_score)

## Compute cosine similarity

In [ ]:
sdf_cross_joined = (
    sdf_cross_joined
    .withColumn('cosine_similarity', compute_cosine_similarity(F.col('array_show'), F.col('array_library'))).cache()
    .drop('array_show', 'array_library')
    .orderBy('time_step', 'id')
)

In [ ]:
sdf_cross_joined.show(5)

## Reduce dataset size by percentile cutoff

In [ ]:
sdf_cross_joined.count()

In [ ]:
similarity_quantile_cutoff = sdf_cross_joined.approxQuantile('cosine_similarity', [percentile_cutoff], approx_quantile_precision)

In [ ]:
similarity_quantile_cutoff

In [ ]:
sdf_cross_joined = (
    sdf_cross_joined
    .where(F.col('cosine_similarity') >= F.lit(similarity_quantile_cutoff[0]))
    .orderBy('time_step', 'id')
)

In [ ]:
sdf_cross_joined.count()

## Aggregate by (timestamp, song_id)

We retain the maximum cosine similarity per (timestamp / song ID) pair:

In [ ]:
sdf_cross_joined.repartition('time_step', 'id')

sdf_agg = (
    sdf_cross_joined
    .groupBy('time_step', 'id')
    .agg(
        F.max('cosine_similarity').alias('cosine_similarity'),
    )
    .orderBy('time_step', F.desc('cosine_similarity'))
)

In [ ]:
sdf_cross_joined.show(5)

## Record the rank per timestamp¶

In [ ]:
sdf_agg.repartition('time_step')

In [ ]:
window_spec = Window.partitionBy('time_step').orderBy(F.desc('cosine_similarity'))

sdf_agg_ranked = (
    sdf_agg
    .orderBy(F.asc('time_step'), F.desc('cosine_similarity'))
    .withColumn('rank', F.row_number().over(window_spec))
)

In [ ]:
sdf_agg_ranked.show(5)

## Keep only the top-ranked rows per time step

In [ ]:
sdf_agg_ranked = (
    sdf_agg_ranked
    .where(F.col('rank') <= 1)
    .drop('rank', 'cosine_similarity')
    .orderBy('time_step')
)

In [ ]:
sdf_agg_ranked.show(5)

## Compute timestamp in seconds

In [ ]:
sdf_ids = (
    sdf_agg_ranked
    .withColumn('timestamp', F.col('time_step') * (hop_length / sampling_rate))
    .drop('time_step')
)

In [ ]:
sdf_ids.show(5)

## Calculate (rough) song intervals

In [ ]:
sdf_ids.repartition('id')

In [ ]:
sdf_ids_agg = (
    sdf_ids
    .groupBy('id')
    .agg(
        F.min('timestamp').alias('p0'),
        F.percentile_approx('timestamp', 0.25).alias('p25'),
        F.percentile_approx('timestamp', 0.75).alias('p75'),
        F.max('timestamp').alias('p100'),
    )
    .orderBy('p0')
)

In [ ]:
sdf_ids_agg.show(10)

## Load the track titles and artists

In [ ]:
sdf_library_names = (
    spark
    .createDataFrame(pd.read_parquet(path_library_parquet))
    .drop('path')
    .orderBy('id')
)

In [ ]:
sdf_library_names.show(5)

## Add artist/title information

In [ ]:
sdf_named = (
    sdf_ids_agg.join(sdf_library_names, on = 'id', how = 'left')
)


In [ ]:
sdf_named.show(10)

## Compute time ranges (IRQ and full)

In [ ]:
sdf_named = (
    sdf_named
    .withColumn('range', F.col('p100') - F.col('p0'))
    .withColumn('IRQ', F.col('p75') - F.col('p25'))
    .orderBy('p25')
    .drop('id')
)

In [ ]:
sdf_named.show(sdf_named.count())

## Filter by IRQ

In [ ]:
sdf_named_cut_min_time_diff = (
    sdf_named
    .where(F.col('IRQ') >= irq_min)
    .where(F.col('IRQ') < irq_max)
    .orderBy('p0')
    .drop('p100', 'IRQ')
    .withColumnRenamed('p25', 'seconds')
    .orderBy('seconds')
)

In [ ]:
sdf_named_cut_min_time_diff.show(5)

## Convert seconds to HH:MM:SS and display

In [ ]:
sdf_named_cut_min_time_diff = (
    sdf_named_cut_min_time_diff
    .withColumn('hour', F.floor(F.col('seconds') / 3600).cast('int'))
    .withColumn('minute', F.floor((F.col('seconds') % 3600) / 60).cast('int'))
    .withColumn('sec', (F.col('seconds') % 60).cast('int'))
    .withColumn('time', F.format_string("%02d:%02d:%02d", 'hour', 'minute', 'sec'))
    .select('artist', 'name', 'time')
    .orderBy('time')
)

In [ ]:
sdf_named_cut_min_time_diff.show(sdf_named_cut_min_time_diff.count())

## Output Mixcloud-compatible text

In [ ]:
result_path = output_directory + '/results_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.txt'

(
    sdf_named_cut_min_time_diff
    .withColumn('text', F.concat(F.col('artist'), F.lit(' - '), F.col('name'), F.lit(' - '), F.col('time')))
    .orderBy('time')
    .select('text')
    .coalesce(1)  
    .write.mode("overwrite").text(result_path)    
)

## Close the Spark session¶

In [ ]:
spark.stop()